# REST API Fundamentals
## What is a REST API?
An API (Application Programming Interface) is a set of rules that allows two systems to communicate.
REST (Representational State Transfer) is the most widely used architectural style for web APIs.

## HTTP Methods & Status Codes
- GET — retrieve data
- POST — create data
- PUT — replace data
- PATCH — partial update
- DELETE — remove data

Status codes: 2xx success | 4xx client error | 5xx server error

In [2]:
import requests

print("requests version:", requests.__version__)

requests version: 2.32.5


In [3]:
url = "https://api.open-meteo.com/v1/forecast?latitude=41.39&longitude=2.15&current_weather=true"

response = requests.get(url)

print("Status code:", response.status_code)
print("Response type:", type(response))

Status code: 200
Response type: <class 'requests.models.Response'>


In [4]:
print(response.text)

{"latitude":41.375,"longitude":2.125,"generationtime_ms":199.11503791809082,"utc_offset_seconds":0,"timezone":"GMT","timezone_abbreviation":"GMT","elevation":59.0,"current_weather_units":{"time":"iso8601","interval":"seconds","temperature":"°C","windspeed":"km/h","winddirection":"°","is_day":"","weathercode":"wmo code"},"current_weather":{"time":"2026-06-04T11:00","interval":900,"temperature":24.2,"windspeed":17.7,"winddirection":209,"is_day":1,"weathercode":3}}


In [5]:
data = response.json()

print(type(data))
print(data)

<class 'dict'>
{'latitude': 41.375, 'longitude': 2.125, 'generationtime_ms': 199.11503791809082, 'utc_offset_seconds': 0, 'timezone': 'GMT', 'timezone_abbreviation': 'GMT', 'elevation': 59.0, 'current_weather_units': {'time': 'iso8601', 'interval': 'seconds', 'temperature': '°C', 'windspeed': 'km/h', 'winddirection': '°', 'is_day': '', 'weathercode': 'wmo code'}, 'current_weather': {'time': '2026-06-04T11:00', 'interval': 900, 'temperature': 24.2, 'windspeed': 17.7, 'winddirection': 209, 'is_day': 1, 'weathercode': 3}}


In [6]:
weather = data['current_weather']

print("Temperature:", weather['temperature'], "°C")
print("Wind speed:", weather['windspeed'], "km/h")
print("Wind direction:", weather['winddirection'], "°")
print("Is daytime:", bool(weather['is_day']))

Temperature: 24.2 °C
Wind speed: 17.7 km/h
Wind direction: 209 °
Is daytime: True


In [7]:
params = {
    "latitude": 41.39,
    "longitude": 2.15,
    "current_weather": True,
    "timezone": "Europe/Madrid"
}

response = requests.get("https://api.open-meteo.com/v1/forecast", params=params)

print("Status code:", response.status_code)
print("URL called:", response.url)

Status code: 200
URL called: https://api.open-meteo.com/v1/forecast?latitude=41.39&longitude=2.15&current_weather=True&timezone=Europe%2FMadrid


In [8]:
def get_weather(latitude, longitude):
    """Fetch current weather for given coordinates."""
    params = {
        "latitude": latitude,
        "longitude": longitude,
        "current_weather": True,
        "timezone": "Europe/Madrid"
    }
    
    response = requests.get("https://api.open-meteo.com/v1/forecast", params=params)
    response.raise_for_status()
    
    return response.json()

# Test with Barcelona
weather_data = get_weather(41.39, 2.15)
print(weather_data['current_weather'])

{'time': '2026-06-04T13:00', 'interval': 900, 'temperature': 24.2, 'windspeed': 17.7, 'winddirection': 209, 'is_day': 1, 'weathercode': 3}


In [9]:
def get_weather_safe(latitude, longitude):
    """Fetch current weather with error handling."""
    params = {
        "latitude": latitude,
        "longitude": longitude,
        "current_weather": True,
        "timezone": "Europe/Madrid"
    }
    
    try:
        response = requests.get("https://api.open-meteo.com/v1/forecast", params=params)
        response.raise_for_status()
        return response.json()
    except requests.exceptions.HTTPError as e:
        print(f"HTTP error: {e}")
    except requests.exceptions.ConnectionError:
        print("Connection error — check your internet connection")
    except requests.exceptions.Timeout:
        print("Request timed out")
    
    return None

# Test with valid coordinates
result = get_weather_safe(41.39, 2.15)
print("Success:", result is not None)

# Test with invalid coordinates
result_bad = get_weather_safe(999, 999)
print("Bad request:", result_bad)

Success: True
HTTP error: 400 Client Error: Bad Request for url: https://api.open-meteo.com/v1/forecast?latitude=999&longitude=999&current_weather=True&timezone=Europe%2FMadrid
Bad request: None
